# E-Commerce Fraud Detection
### End-to-End Data Science Project — Security Intelligence Context

This notebook walks through the full fraud detection pipeline from raw transaction data to a scored, business-ready output. The goal is to build a model that maximizes net value to a security operations team — not just raw accuracy.

**Pipeline:**
1. Data overview
2. Exploratory data analysis (EDA)
3. Hypothesis validation
4. Feature engineering
5. XGBoost modeling
6. Model explainability (SHAP)
7. Threshold optimization via cost-benefit analysis
8. Final results & business impact


## Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="darkgrid")

df = pd.read_csv("../data/transactions.csv")
df["transaction_time"] = pd.to_datetime(df["transaction_time"])
df["hour"] = df["transaction_time"].dt.hour
df["day_of_week"] = df["transaction_time"].dt.dayofweek
df["month"] = df["transaction_time"].dt.month
print(f"Loaded {len(df):,} transactions")


---
## 1. Data Overview

The dataset contains 299,695 e-commerce transactions with 17 features covering transaction metadata, user history, security check outcomes, and geography. The target variable is `is_fraud`.

The class imbalance is significant at **2.21% fraud rate** — meaning for every fraud case, there are ~44 legitimate transactions. This will directly inform our modeling approach.


In [ ]:
print("=== SHAPE ===")
print(df.shape)

print("\n=== NULL VALUES ===")
print(df.isnull().sum())

print("\n=== FRAUD RATE ===")
fraud_rate = df["is_fraud"].mean() * 100
print(f"{fraud_rate:.2f}% of transactions are fraudulent")
print(f"Fraud: {df['is_fraud'].sum():,} | Legit: {(df['is_fraud']==0).sum():,}")

df.head()


---
## 2. Exploratory Data Analysis

The goal of EDA is to identify which features have the most signal for fraud — both individually and in combination. We look at geography, merchant category, timing, transaction amounts, and security flags.


### 2.1 Fraud Rate by Country

Some countries show consistently higher fraud rates. The more actionable insight, though, is *card country vs. transaction country mismatch* — engineered in Section 4.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
fraud_by_country = df.groupby("country")["is_fraud"].mean().sort_values(ascending=False) * 100
fraud_by_country.plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Fraud Rate by Country (%)")
ax.set_xlabel("Country")
ax.set_ylabel("Fraud Rate (%)")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()
print(fraud_by_country.round(2))


### 2.2 Fraud Rate by Merchant Category

Merchant category shows no meaningful differentiation — all categories cluster within 1–2% of the baseline. Fraud is not category-specific; it is account and behavior-specific.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
fraud_by_merchant = df.groupby("merchant_category")["is_fraud"].mean().sort_values(ascending=False) * 100
fraud_by_merchant.plot(kind="bar", ax=ax, color="coral")
ax.set_title("Fraud Rate by Merchant Category (%)")
ax.set_xlabel("Merchant Category")
ax.set_ylabel("Fraud Rate (%)")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()
print(fraud_by_merchant.round(2))


### 2.3 Fraud Rate by Hour of Day

Fraud is relatively uniform across hours — no strong time-of-day signal. `hour` may still contribute marginal value as a model feature but will not carry significant weight.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
fraud_by_hour = df.groupby("hour")["is_fraud"].mean().sort_index() * 100
fraud_by_hour.plot(kind="line", ax=ax, color="crimson", linewidth=5, marker="o", markersize=8)
ax.set_title("Fraud Rate by Hour of Day (%)")
ax.set_xlabel("Hour (0 = midnight, 12 = noon)")
ax.set_ylabel("Fraud Rate (%)")
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.show()


### 2.4 Transaction Amount: Fraud vs. Legitimate

Fraudulent transactions average **$590 vs. $168** for legitimate — a 3.5x difference. This makes transaction amount a strong signal, especially when expressed as a ratio to the user's own history.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
df[df["is_fraud"] == 0]["amount"].hist(bins=50, alpha=0.5, label="Legit", color="steelblue", ax=ax)
df[df["is_fraud"] == 1]["amount"].hist(bins=50, alpha=0.5, label="Fraud", color="crimson", ax=ax)
ax.set_title("Transaction Amount Distribution: Fraud vs. Legit")
ax.set_xlabel("Amount ($)")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.show()
print(df.groupby("is_fraud")["amount"].describe().round(2))


### 2.5 Security Flag Analysis

Security flags are the most powerful individual fraud signals in the dataset:

| Flag | Fraud Multiplier |
|---|---|
| CVV fail | **20x** baseline |
| AVS fail | **18x** baseline |
| 3D Secure fail | ~4x baseline |

These signals are strong individually — and even stronger in combination, as shown in Section 3.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, col in zip(axes, ["avs_match", "cvv_result", "three_ds_flag"]):
    fraud_by_flag = df.groupby(col)["is_fraud"].mean() * 100
    fraud_by_flag.plot(kind="bar", ax=ax, color=["crimson", "steelblue"])
    ax.set_title(f"Fraud Rate by {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Fraud Rate (%)")
    ax.set_xticklabels(["Failed (0)", "Passed (1)"], rotation=0)
plt.tight_layout()
plt.show()


### 2.6 Correlation Heatmap

`shipping_distance_km` has the strongest positive correlation with fraud. Security flags (`avs_match`, `cvv_result`, `three_ds_flag`) are negatively correlated — failing these checks predicts fraud. `account_age_days` and `total_transactions_user` are also meaningful: newer accounts with little history are higher risk.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
numeric_cols = ["amount", "avg_amount_user", "account_age_days",
                "total_transactions_user", "shipping_distance_km",
                "avs_match", "cvv_result", "three_ds_flag",
                "promo_used", "hour", "day_of_week", "month", "is_fraud"]
corr_matrix = df[numeric_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, ax=ax, linewidths=0.5)
ax.set_title("Correlation Heatmap — Numeric Features vs. is_fraud")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()


---
## 3. Hypothesis Validation

EDA shows each security flag is a strong individual signal. The hypothesis: **when multiple risk factors appear together, fraud rate compounds dramatically.** We test this directly before encoding it as a feature.


In [ ]:
high_risk = df[
    (df["cvv_result"] == 0) &
    (df["avs_match"] == 0) &
    (df["three_ds_flag"] == 0) &
    (df["amount"] > df["amount"].quantile(0.75)) &
    (df["account_age_days"] < df["account_age_days"].quantile(0.25))
]

overall_rate = df["is_fraud"].mean() * 100
high_risk_rate = high_risk["is_fraud"].mean() * 100

print(f"Overall fraud rate:          {overall_rate:.2f}%")
print(f"High-risk group fraud rate:  {high_risk_rate:.2f}%")
print(f"Multiplier:                  {high_risk_rate / overall_rate:.1f}x more likely")
print(f"High-risk transactions:      {len(high_risk):,} out of {len(df):,}")


**Result:** The high-risk group hits a **44.71% fraud rate** — 20x the 2.21% baseline. This validates the hypothesis: combining signals is far more predictive than any flag alone. These combinations become engineered features in the next section.


---
## 4. Feature Engineering

We engineer 6 rule-based features that explicitly encode the fraud patterns identified in EDA. This gives the model pre-computed signals rather than requiring it to rediscover these combinations from raw data.

| Feature | Description |
|---|---|
| `security_fails` | Count of failed security checks (0–3) |
| `amount_vs_avg` | Transaction amount ÷ user historical average |
| `country_mismatch` | Card country ≠ transaction country |
| `all_security_failed` | All 3 checks failed simultaneously |
| `mismatch_and_security_fail` | Country mismatch AND ≥2 security failures |
| `amount_spike` | Transaction ≥3× user average spend |


In [ ]:
# ── 1. Country mismatch ────────────────────────────────────────
df["country_mismatch"] = (df["country"] != df["bin_country"]).astype(int)

# ── 2. Security fails score (0-3) ──────────────────────────────
df["security_fails"] = (
    (1 - df["avs_match"]) +
    (1 - df["cvv_result"]) +
    (1 - df["three_ds_flag"])
)

# ── 3. Amount vs. user average ─────────────────────────────────
df["amount_vs_avg"] = df["amount"] / (df["avg_amount_user"] + 1)

# ── 4. All security checks failed ──────────────────────────────
df["all_security_failed"] = (
    (df["cvv_result"] == 0) &
    (df["avs_match"] == 0) &
    (df["three_ds_flag"] == 0)
).astype(int)

# ── 5. Country mismatch + 2+ security failures ─────────────────
df["mismatch_and_security_fail"] = (
    (df["country_mismatch"] == 1) &
    (df["security_fails"] >= 2)
).astype(int)

# ── 6. Amount spike vs user normal ─────────────────────────────
df["amount_spike"] = (df["amount_vs_avg"] > 3).astype(int)

# ── Encode categorical columns ──────────────────────────────────
df = pd.get_dummies(df, columns=["country", "bin_country", "channel", "merchant_category"])

print("Feature engineering complete. New shape:", df.shape)


#### Rule Validation


In [ ]:
rules = [
    ("all_security_failed", "All 3 security checks failed"),
    ("mismatch_and_security_fail", "Country mismatch + 2+ security fails"),
    ("amount_spike", "Amount >= 3x user average"),
]

baseline = df["is_fraud"].mean() * 100
print(f"Baseline fraud rate: {baseline:.2f}%\n")
print(f"{'Feature':<40} {'Fraud Rate':>10} {'Count':>10} {'Multiplier':>12}")
print("-" * 75)
for col, label in rules:
    group = df[df[col] == 1]
    rate = group["is_fraud"].mean() * 100
    print(f"{label:<40} {rate:>9.1f}% {len(group):>10,} {rate/baseline:>11.1f}x")


---
## 5. Machine Learning Model

We use **XGBoost** with `scale_pos_weight=44` to handle the 44:1 class imbalance. This instructs the model to penalize missed fraud cases 44x more heavily than missed legitimate transactions during training.

The 80/20 stratified train/test split preserves the fraud rate in both sets.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier

X = df.drop(columns=["transaction_id", "user_id", "transaction_time", "is_fraud"])
y = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set:     {X_test.shape}")
print(f"Fraud in train: {y_train.sum():,} ({y_train.mean()*100:.2f}%)")
print(f"Fraud in test:  {y_test.sum():,} ({y_test.mean()*100:.2f}%)")


In [ ]:
scale = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale:.1f}")

model = XGBClassifier(
    n_estimators=500,       # more trees for more learning opportunities
    max_depth=8,            # captures complex interaction patterns
    learning_rate=0.03,     # slow, careful corrections
    subsample=0.8,          # 80% of rows per tree to reduce overfitting
    colsample_bytree=0.8,   # 80% of features per tree
    min_child_weight=5,     # min 5 samples per leaf
    gamma=0.1,              # min gain required to split
    scale_pos_weight=scale,
    random_state=42,
    eval_metric="aucpr"
)

print("Training model...")
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("\n=== MODEL RESULTS (default threshold 0.5) ===")
print(classification_report(y_test, y_pred, target_names=["Legit", "Fraud"]))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")


---
## 6. Model Explainability — SHAP

SHAP (SHapley Additive exPlanations) shows which features drove each prediction and in which direction. In a security context this is not optional — analysts need to understand *why* a transaction was flagged.

**3 of the top 8 SHAP features are engineered features**, confirming that domain knowledge added real signal beyond raw data.


In [ ]:
import shap

feature_names = {
    "shipping_distance_km": "Shipping Distance (km)",
    "account_age_days": "Account Age (days)",
    "avs_match": "AVS Match",
    "channel_app": "Channel: App",
    "amount": "Transaction Amount",
    "security_fails": "Security Failures (0-3)",
    "amount_vs_avg": "Amount vs User Average",
    "country_mismatch": "Card/Transaction Country Mismatch",
    "avg_amount_user": "User Average Spend",
    "total_transactions_user": "Total User Transactions",
    "hour": "Hour of Day",
    "channel_web": "Channel: Web",
    "cvv_result": "CVV Result",
    "three_ds_flag": "3D Secure Flag",
    "high_ship_security_fail": "High Shipping + Security Fail",
}

X_test_sample = X_test.iloc[:2000].rename(columns=feature_names)

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_sample)

plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_test_sample, max_display=20, show=False)
plt.title("SHAP Feature Importance — Top 20 Fraud Predictors")
plt.tight_layout()
plt.show()


**Fraudster profile from SHAP:**
- New account with low transaction history
- Transaction amount significantly above personal average (`amount_vs_avg` spike)
- Long shipping distance
- Failed 2+ security checks
- Often uses web channel


---
## 7. Threshold Optimization

The default 0.5 threshold optimizes for balanced accuracy — not business value. The right question is: **at which threshold does the model generate the most net dollar value?**

**Assumptions:**
- Average fraud transaction value: **$590**
- Analyst review: **15 min @ $35/hr = $8.75 per flagged transaction**
- Net value = (fraud caught × $590) − (total flags × $8.75)

Each false alarm costs $8.75. Each caught fraud saves $590. The model should flag aggressively until the marginal analyst cost exceeds the marginal fraud savings.


In [ ]:
avg_fraud_amount = 590
analyst_hourly_rate = 35
review_time_hours = 15 / 60

print(f"{'Threshold':<10} {'Caught':>8} {'Missed':>8} {'FalseAlarm':>12} {'FraudSaved':>12} {'AnalystCost':>13} {'NetValue':>12}")
print("-" * 90)

thresholds = [0.05, 0.08, 0.10, 0.12, 0.15, 0.20, 0.30, 0.50, 0.70, 0.92]
for threshold in thresholds:
    y_pred_t = (y_prob > threshold).astype(int)
    caught = ((y_pred_t == 1) & (y_test == 1)).sum()
    missed = ((y_pred_t == 0) & (y_test == 1)).sum()
    false_alarms = ((y_pred_t == 1) & (y_test == 0)).sum()
    total_reviews = y_pred_t.sum()
    fraud_saved = caught * avg_fraud_amount
    analyst_cost = total_reviews * review_time_hours * analyst_hourly_rate
    net_value = fraud_saved - analyst_cost
    marker = " <-- optimal" if threshold == 0.12 else ""
    print(f"{threshold:<10} {caught:>8,} {missed:>8,} {false_alarms:>12,} ${fraud_saved:>10,} ${analyst_cost:>11,.0f} ${net_value:>10,.0f}{marker}")

print(f"\nBased on test set ({len(y_test):,} transactions)")


---
## 8. Final Results & Business Impact

**Threshold 0.12** is the optimal operating point. At this threshold, 94% of fraud cases are caught. The 80% false positive rate is acceptable because each review costs only $8.75 while each caught fraud saves $590 — a 67:1 return on analyst time.


In [ ]:
OPTIMAL_THRESHOLD = 0.12

y_pred_final = (y_prob > OPTIMAL_THRESHOLD).astype(int)

caught = ((y_pred_final == 1) & (y_test == 1)).sum()
missed = ((y_pred_final == 0) & (y_test == 1)).sum()
false_alarms = ((y_pred_final == 1) & (y_test == 0)).sum()
total_reviews = y_pred_final.sum()

fraud_saved = caught * avg_fraud_amount
analyst_cost = total_reviews * review_time_hours * analyst_hourly_rate
net_value = fraud_saved - analyst_cost

print("=== FINAL MODEL — THRESHOLD 0.12 ===")
print(classification_report(y_test, y_pred_final, target_names=["Legit", "Fraud"]))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}")

print(f"\n=== DETECTION PERFORMANCE ===")
print(f"Fraud caught:    {caught:,} of {y_test.sum():,} ({caught/y_test.sum()*100:.1f}%)")
print(f"Fraud missed:    {missed:,} ({missed/y_test.sum()*100:.1f}%)")
print(f"False alarms:    {false_alarms:,}")

print(f"\n=== FINANCIAL IMPACT (per 60k transactions) ===")
print(f"Fraud value recovered: ${fraud_saved:,}")
print(f"Analyst review cost:   ${analyst_cost:,.0f}")
print(f"Net value:             ${net_value:,.0f}")
print(f"Annual estimate (x5):  ${net_value*5:,.0f}")


---
## 9. Export for Dashboard

Score all 299,695 transactions and export with risk tiers for the Tableau monitoring dashboard.


In [ ]:
X_full = df.drop(columns=["transaction_id", "user_id", "transaction_time", "is_fraud"])
full_probs = model.predict_proba(X_full)[:, 1]
full_preds = (full_probs > OPTIMAL_THRESHOLD).astype(int)

scored = pd.DataFrame({
    "transaction_id": df["transaction_id"],
    "user_id": df["user_id"],
    "transaction_time": df["transaction_time"],
    "amount": df["amount"],
    "shipping_distance_km": df["shipping_distance_km"],
    "account_age_days": df["account_age_days"],
    "security_fails": df["security_fails"],
    "country_mismatch": df["country_mismatch"],
    "fraud_probability": full_probs.round(4),
    "predicted_fraud": full_preds,
    "actual_fraud": df["is_fraud"],
    "risk_tier": pd.cut(full_probs, bins=[0, 0.25, 0.5, 0.75, 1.0],
                        labels=["Low", "Medium", "High", "Critical"])
})

scored.to_csv("../outputs/scored_transactions.csv", index=False)
print(f"Exported {len(scored):,} scored transactions")
print(f"\nRisk tier distribution:")
print(scored["risk_tier"].value_counts().sort_index())
